In [10]:
import os
import re
import json
import math
import sys
import gc
import csv
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from pyvis.network import Network
import igraph as ig
import spacy
import warnings
warnings.filterwarnings("ignore")



## Preparar la data y envs

In [11]:
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("vader_lexicon", quiet=True)

DATASET = "traficogt.txt" # cambiar a 'tioberny.txt'
OUT_DIR = Path("processed_output")
OUT_DIR.mkdir(exist_ok=True)

# stopwords en esp y eng por si acaso
SW_ES = set(stopwords.words('spanish'))
SW_EN = set(stopwords.words('english'))

In [18]:
SPACY_ES = None
try:
    SPACY_ES = spacy.load('es_core_news_sm')
except Exception:
    SPACY_ES = None

sia = SentimentIntensityAnalyzer()

RE_URL = re.compile(r'https?://\S+|www\.\S+')
RE_USER = re.compile(r'@([A-Za-z0-9_]+)')
RE_HASHTAG = re.compile(r'#(\w+)')
RE_EMOJI = re.compile("[\U00010000-\U0010ffff]", flags=re.UNICODE)
RE_PUNCT = re.compile(r"[\.,;:\!?\(\)\[\]{}<>\"'”“–——-]")

def normalize_username(u):
    if u is None:
        return None
    u = str(u).strip()
    u = u.lower()
    if u.startswith('@'):
        u = u[1:]
    return u

def strip_accents(text):
    # normalizar los acentos para consistencia
    text = unicodedata.normalize('NFKD', text)
    return ''.join([c for c in text if not unicodedata.combining(c)])

def clean_text(text, remove_urls=True, remove_mentions=False, remove_hashtags=True,
               remove_emojis=True, lower=True, remove_punct=True, remove_numbers=False,
               remove_stopwords=True, lang_stopwords='both', lemmatize=False):
    if text is None:
        return "", []
    txt = str(text)

    if remove_urls:
        txt = RE_URL.sub('', txt)
    if remove_emojis:
        txt = RE_EMOJI.sub('', txt)
    if lower:
        txt = txt.lower()
    txt = strip_accents(txt)
    if remove_mentions:
        txt = RE_USER.sub('', txt)
    if remove_hashtags:
        txt = RE_HASHTAG.sub(lambda m: m.group(1), txt)
    if remove_punct:
        txt = RE_PUNCT.sub(' ', txt)
    if remove_numbers:
        txt = re.sub(r'\d+', ' ', txt)

    tokens = word_tokenize(txt)

    if remove_stopwords:
        sw = set()
        if lang_stopwords in ('es', 'both'):
            sw |= SW_ES
        if lang_stopwords in ('en', 'both'):
            sw |= SW_EN
        tokens = [t for t in tokens if t not in sw and len(t) > 1]

    if lemmatize and SPACY_ES is not None:
        doc = SPACY_ES(' '.join(tokens))
        tokens = [tok.lemma_ for tok in doc]


    cleaned = ' '.join(tokens)
    return cleaned, tokens


In [20]:
# carga tweets desde un archivo json. soporta json array o json por linea.
def load_json_tweets(path: Path):
    tweets = []
    text = path.read_text(encoding='utf-8', errors='ignore')
    text = text.strip()
    if not text:
        return tweets
    try:
        if text[0] == '[':
            arr = json.loads(text)
            return arr
    except Exception:
        pass

    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                tweets.append(obj)
            except Exception:
                try:
                    start = line.find('{')
                    obj = json.loads(line[start:])
                    tweets.append(obj)
                except Exception:
                    continue
    return tweets

# extrae menciones, hashtags y urls de un tweet.
def extract_entities(tweet):
    mentions = RE_USER.findall(tweet.get('text', ''))
    hashtags = RE_HASHTAG.findall(tweet.get('text', ''))
    urls = RE_URL.findall(tweet.get('text', ''))
    return {
        'mentions': [normalize_username(m) for m in mentions],
        'hashtags': hashtags,
        'urls': urls
    }

# construye un dataframe a partir de la lista de objetos tweet.
def build_dataframe(tweets_raw):
    data = []
    for tw in tweets_raw:
        tweet_id = tw.get('id')
        text = tw.get('text')
        created_at_str = tw.get('created_at')
        author_id = tw.get('author_id')
        in_reply_to_user_id = tw.get('in_reply_to_user_id')

        is_retweet = False
        is_reply = False
        referenced_tweet_id = None
        if 'referenced_tweets' in tw and tw['referenced_tweets']:
            for ref_tweet in tw['referenced_tweets']:
                if ref_tweet.get('type') == 'retweeted':
                    is_retweet = True
                    referenced_tweet_id = ref_tweet.get('id')
                elif ref_tweet.get('type') == 'replied_to':
                    is_reply = True
                    referenced_tweet_id = ref_tweet.get('id')

        cleaned_text, tokens = clean_text(text, remove_mentions=True, remove_hashtags=True, remove_urls=True)
        entities = extract_entities(tw)
        sentiment_score = sia.polarity_scores(cleaned_text)['compound']

        data.append({
            'id': tweet_id,
            'text': text,
            'cleaned_text': cleaned_text,
            'tokens': tokens,
            'created_at': created_at_str,
            'author_id': author_id,
            'in_reply_to': in_reply_to_user_id,
            'is_retweet': is_retweet,
            'is_reply': is_reply,
            'referenced_tweet_id': referenced_tweet_id,
            'mentions': entities['mentions'],
            'hashtags': entities['hashtags'],
            'urls': entities['urls'],
            'sentiment_score': sentiment_score
        })

    df = pd.DataFrame(data)

    try:
        df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', utc=True)
    except Exception as e:
        print(f"error converting 'created_at' to datetime: {e}")
        try:
             df['created_at'] = pd.to_datetime(df['created_at'], format='%a %b %d %H:%M:%S +0000 %Y', errors='coerce', utc=True)
        except Exception as e2:
             print(f"error converting 'created_at' to datetime with alternative format: {e2}")
             df['created_at'] = pd.NA

    df = df.drop_duplicates(subset=['id'])
    df = df.dropna(subset=['id'])

    return df

# construye un grafo de interacciones (menciones, retweets, replies).
def build_interaction_graph(df: pd.DataFrame, min_edge_weight=1):
    edges = {}

    for _, row in df.iterrows():
        src = row['author_id']
        if src is None:
            continue

        for mention in row['mentions']:
            tgt = normalize_username(mention)
            if tgt and src != tgt:
                key = (src, tgt, 'mention')
                edges[key] = edges.get(key, 0) + 1

        if row['is_retweet'] and row['referenced_tweet_id']:
            original_tweet = df[df['id'] == row['referenced_tweet_id']]
            if not original_tweet.empty:
                tgt = original_tweet.iloc[0]['author_id']
                if tgt and src != tgt:
                    key = (src, tgt, 'retweet')
                    edges[key] = edges.get(key, 0) + 1

        reply_target_id = None
        if row['in_reply_to']:
            reply_target_id = row['in_reply_to']
        elif row['is_reply'] and row['referenced_tweet_id']:
             original_tweet = df[df['id'] == row['referenced_tweet_id']]
             if not original_tweet.empty:
                 reply_target_id = original_tweet.iloc[0]['author_id']

        if reply_target_id:
            tgt = normalize_username(reply_target_id)
            if tgt and src != tgt:
                key = (src, tgt, 'reply')
                edges[key] = edges.get(key, 0) + 1

    edge_rows = []
    for (s, t, typ), w in edges.items():
        if w >= min_edge_weight:
            edge_rows.append({'source': s, 'target': t, 'type': typ, 'weight': w})
    edf = pd.DataFrame(edge_rows)

    G = nx.DiGraph()
    if not edf.empty:
        for _, r in edf.iterrows():
            G.add_node(r['source'])
            G.add_node(r['target'])
            if G.has_edge(r['source'], r['target']):
                G[r['source']][r['target']]['weight'] += int(r['weight'])
                G[r['source']][r['target']]['types'].add(r['type'])
            else:
                G.add_edge(r['source'], r['target'], weight=int(r['weight']), types=set([r['type']]))

    return G, edf

# carga datos, procesa, construye grafo y guarda resultados.
def main(dataset_path: str):
    path = Path(dataset_path)
    if not path.exists():
        raise FileNotFoundError(f"archivo no encontrado: {dataset_path}")

    print(f"cargando tweets desde {path}...")
    tweets_raw = load_json_tweets(path)
    print(f"{len(tweets_raw)} objetos json leidos (aprox).")

    print("construyendo dataframe procesado...")
    df = build_dataframe(tweets_raw)
    print(f"dataframe con {len(df)} registros tras limpieza y eliminacion de duplicados.")

    proc_csv = OUT_DIR / (path.stem + '_processed.csv')
    df.to_csv(proc_csv, index=False)
    print(f"procesado guardado en: {proc_csv}")

    print("construyendo grafo de interacciones...")
    G, edf = build_interaction_graph(df)
    print(f"grafo con {G.number_of_nodes()} nodos y {G.number_of_edges()} aristas.")

    edf_csv = OUT_DIR / (path.stem + '_edges.csv')
    edf.to_csv(edf_csv, index=False)
    print(f"lista de aristas guardada en: {edf_csv}")

    print("calculando metricas de grafo...")
    indeg = {n: sum([edata.get('weight', 1) for _, _, edata in G.in_edges(n, data=True)]) for n in G.nodes()}
    outdeg = {n: sum([edata.get('weight', 1) for _, _, edata in G.out_edges(n, data=True)]) for n in G.nodes()}
    for n in G.nodes():
        indeg[n] = indeg.get(n, 0)
        outdeg[n] = outdeg.get(n, 0)

    deg_df = pd.DataFrame([{'user': n, 'in_weight': indeg[n], 'out_weight': outdeg[n]} for n in G.nodes()])
    top_in = deg_df.sort_values('in_weight', ascending=False).head(10)
    top_out = deg_df.sort_values('out_weight', ascending=False).head(10)

    top_in.to_csv(OUT_DIR / (path.stem + '_top_in.csv'), index=False)
    top_out.to_csv(OUT_DIR / (path.stem + '_top_out.csv'), index=False)

    print('top 10 usuarios por interacciones recibidas (in_weight):')
    print(top_in)
    print('\ntop 10 usuarios por interacciones enviadas (out_weight):')
    print(top_out)

    graph_path = OUT_DIR / (path.stem + '_graph.graphml')
    nx.write_graphml(G, graph_path)
    print(f"grafo guardado en: {graph_path}")

    return df, G, edf